# Spatial machine learning analysis of home delivery in Ethiopia (EDHS 2024–25)
**Google Colab version (revised).** Every table and figure is displayed in the notebook *first*, then saved to your Drive. Each figure is a separate single-panel image.

**Before running:** create a Drive folder (default `MyDrive/EDHS2024`) containing `data_3.csv`, `ETGC8AFL.csv` and all `ETGE8AFL.*` shapefile parts (`.shp .shx .dbf .prj .cpg .sbn .sbx`). Results are written to `MyDrive/EDHS2024/outputs`.

*Optional:* place an official admin-1 boundary file with the 2023 regions as `eth_admin1_official.gpkg` (column `reg` holding the 14 region names used here) in the same folder; it is used automatically instead of the approximate boundaries. Run all cells (Runtime ▸ Run all); the full run takes roughly 30–45 minutes on a free Colab CPU because of nested tuning.

In [ ]:
# 1. Install packages
!pip -q install geopandas pyogrio shapely libpysal esda shap lightgbm xgboost statsmodels

In [ ]:
# 2. Mount Google Drive and set paths  --> EDIT DATA_DIR if your folder is named differently
import os
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/EDHS2024'
except ImportError:
    DATA_DIR = os.environ.get('EDHS_DIR', '.')
OUT = os.path.join(DATA_DIR, 'outputs'); os.makedirs(OUT, exist_ok=True)
P = lambda name: os.path.join(OUT, name)
need = ['data_3.csv','ETGC8AFL.csv','ETGE8AFL.shp','ETGE8AFL.shx','ETGE8AFL.dbf']
for f in need: print(('OK      ' if os.path.exists(os.path.join(DATA_DIR,f)) else 'MISSING ')+f)
assert all(os.path.exists(os.path.join(DATA_DIR,f)) for f in need), 'Put the data files in DATA_DIR first'

In [ ]:
# 3. Imports, versions and display/save helpers
import warnings; warnings.filterwarnings('ignore')
import sys, json, time, importlib
import numpy as np, pandas as pd, geopandas as gpd, statsmodels.api as sm
import matplotlib; import matplotlib.pyplot as plt; import matplotlib.patheffects as pe
from IPython.display import display
np.random.seed(1)
print('Python', sys.version.split()[0])
for m in ['numpy','pandas','geopandas','shapely','statsmodels','sklearn','xgboost','lightgbm','shap','libpysal','esda','matplotlib','scipy']:
    try: print(f'{m:12s}', importlib.import_module(m).__version__)
    except Exception as e: print(m,'n/a')
def show_table(df, name, title=None, index=True):
    "Print/display a table first, then save it as CSV."
    if title: print('\n'+title)
    display(df); df.to_csv(P(name), index=index); print(f'-> saved {P(name)}')
def show_fig(fig, name, dpi=300):
    "Display a figure in the notebook first, then save it as PNG."
    display(fig); fig.savefig(P(name), dpi=dpi, bbox_inches='tight'); plt.close(fig); print(f'-> saved {P(name)}')
res = {}; halo=[pe.withStroke(linewidth=2.5,foreground='white')]

## A. Analytic dataset
Unit of analysis: the **most recent live birth in the five years before interview** (births-file record `m80 == 1`, `p19 < 60`). Outcome: **home delivery** (`m15` = 11 respondent's home or 12 other home) vs facility delivery (codes 21–46). Place "other" (96) is excluded.

In [ ]:
d = pd.read_csv(os.path.join(DATA_DIR,'data_3.csv'), low_memory=False)
print('Pregnancy-level file:', d.shape, '| women:', d.caseid.nunique(), '| clusters:', d.v001.nunique())
b = d[(d.m80==1) & (d.p19<60)].copy()
print('Most recent live births in last 5 years:', len(b)); print(b.m15.value_counts().sort_index().to_string())
b = b[b.m15!=96].copy()
b['home'] = b.m15.isin([11,12]).astype(int); b['w'] = b.v005/1e6
cv = pd.read_csv(os.path.join(DATA_DIR,'ETGC8AFL.csv'))
g = gpd.read_file(os.path.join(DATA_DIR,'ETGE8AFL.shp'))[['DHSCLUST','LATNUM','LONGNUM','URBAN_RURA','DHSREGNA']]
cvk = ['Elevation','Global_Human_Footprint','Travel_Times','Nightlights_Composite','All_Population_Count_2020','Rainfall_2020','Enhanced_Vegetation_Index_2020',
       'Land_Surface_Temperature_2020','Aridity_2020','Malaria_Prevalence_2020','Livestock_Cattle','Drought_Episodes','Growing_Season_Length']
cvk = [c for c in cvk if c in cv.columns]
b = b.merge(cv[['DHSCLUST']+cvk].merge(g,on='DHSCLUST'), left_on='v001', right_on='DHSCLUST', how='left')
X = pd.DataFrame(index=b.index)
X['age']=b.v012; X['educ']=b.v106; X['wealth']=b.v190; X['rural']=(b.v025==2).astype(int); X['region']=b.v024
X['religion']=b.v130; X['married']=b.v501.isin([1,2]).astype(int); X['parity']=b.v201.clip(upper=10); X['age_first_birth']=b.v212
X['anc_visits']=b.m14.replace(98,np.nan).clip(upper=12); X['anc4']=(X.anc_visits>=4).astype(float).where(X.anc_visits.notna())
X['wanted']=b.m10; X['news']=b.v157; X['radio']=b.v158; X['tv']=b.v159; X['insured']=b.v481
# self-reported minutes to the nearest health facility (DHS variable v483a); '600+' is top-coded; we cap at 300 min
X['dist_min']=b.v483a.clip(upper=300); X['hh_size']=b.v136; X['female_head']=(b.v151==2).astype(int)
X['bank']=b.v170; X['mobile']=b.v169a; X['working']=b.v714; X['health_decision']=b.v743a; X['sex_child']=b.p4; X['tetanus']=b.m1.replace(8,np.nan)
for c in cvk: X[c]=b[c]
X['lat']=b.LATNUM; X['lon']=b.LONGNUM; X['cluster']=b.v001; X['home']=b.home; X['w']=b.w; X['stratum']=b.v022; X['psu']=b.v021
# Cluster contextual variables: share of the SAMPLED MOTHERS (analytic births) in the cluster in the poorest two wealth quintiles / with no education.
# They use only wealth and education of respondents (never the outcome) and so are available for any new cluster that is surveyed.
X['c_noeduc']=b.groupby('v001').v106.transform(lambda s:(s==0).mean()); X['c_poor']=b.groupby('v001').v190.transform(lambda s:(s<=2).mean())
X['c_n']=b.groupby('v001').v001.transform('size')
# Sensitivity version: all interviewed women in the cluster, leave-one-out (excludes the woman herself)
wm = d.drop_duplicates('caseid')[['caseid','v001','v190','v106']]
agg = wm.groupby('v001').agg(n=('caseid','size'), poor=('v190',lambda s:(s<=2).sum()), noed=('v106',lambda s:(s==0).sum()))
t = b[['caseid','v001','v190','v106']].join(agg, on='v001')
X['c_poor_loo'] = ((t.poor-(t.v190<=2))/(t.n-1)).values; X['c_noeduc_loo'] = ((t.noed-(t.v106==0))/(t.n-1)).values
y = X.home.values; w = X.w.values
res.update(n=int(len(X)), n_clusters=int(X.cluster.nunique()), prev_w=float(np.average(y,weights=w)), prev_u=float(y.mean()))
print(f"\nAnalytic sample: {res['n']} births, {res['n_clusters']} clusters | home delivery weighted {100*res['prev_w']:.1f}% / unweighted {100*res['prev_u']:.1f}%")
print('Births per cluster: median', X.groupby('cluster').size().median(), '| min', X.groupby('cluster').size().min(), '| max', X.groupby('cluster').size().max())
print('Travel-time variable v483a (self-reported minutes): missing', int(b.v483a.isna().sum()), '| top-coded 600+:', int((b.v483a==600).sum()), '| capped at 300:', int((b.v483a>=300).sum()),
      '| median', b.v483a.median(), 'IQR', b.v483a.quantile([.25,.75]).tolist())
display(X.head())

### A2. Missing data for every candidate predictor, and design-based estimators
Preprocessing rules: logistic regression and random forest use **median imputation fitted inside each training fold** (scikit-learn pipelines); XGBoost and LightGBM handle missing values natively. Categorical variables are numeric codes for tree models and dummy-coded (region, religion) for logistic regression.

In [ ]:
feat = ['age','educ','wealth','rural','region','religion','married','parity','age_first_birth','anc_visits','wanted','news','radio','tv','insured','dist_min','hh_size','female_head','bank','mobile','working','health_decision','sex_child','tetanus']+cvk+['c_noeduc','c_poor']
miss = (X[feat].replace({'Travel_Times':{-9999:np.nan}}).isna().mean()*100).round(2).rename('% missing').to_frame(); miss['n missing']=(miss['% missing']/100*len(X)).round().astype(int)
show_table(miss.sort_values('% missing',ascending=False),'table_s1_missingness.csv','Missingness of all candidate predictors (n = %d births)'%len(X))
cc = X[feat].replace({'Travel_Times':{-9999:np.nan}}).notna().all(axis=1); print('Complete-case births:', int(cc.sum()), f'({100*cc.mean():.1f}%)')
from scipy import stats
def _psu_totals(Z, strata, psu):
    df_ = pd.DataFrame(Z); df_['_s']=np.asarray(strata); df_['_p']=np.asarray(psu)
    return df_.groupby(['_s','_p']).sum()
def taylor_cov(Z, strata, psu):
    "Taylor-series linearised variance of weighted totals of Z (stratified, single-stage PSU sampling, with-replacement approximation)."
    tot=_psu_totals(Z,strata,psu); k=Z.shape[1]; B=np.zeros((k,k)); nh_tot=0; lonely=0
    for s,grp in tot.groupby(level=0):
        n=len(grp); nh_tot+=1
        if n<2: lonely+=1; continue
        dev=grp.values-grp.values.mean(0); B+=n/(n-1)*dev.T@dev
    return B, tot.shape[0], nh_tot, lonely
def svy_mean(yv, wv, strata, psu):
    p=np.sum(wv*yv)/np.sum(wv); z=(wv*(yv-p)/np.sum(wv))[:,None]; B,npsu,nstr,_=taylor_cov(z,strata,psu); se=float(np.sqrt(B[0,0])); df_=max(npsu-nstr,1); tq=stats.t.ppf(.975,df_)
    return p, se, p-tq*se, p+tq*se
def svy_logit(Z, yv, wv, strata, psu):
    Zc=sm.add_constant(Z); fit=sm.GLM(yv,Zc,family=sm.families.Binomial(),freq_weights=wv).fit(); mu=fit.fittedvalues.values; Xm=Zc.values
    score=(wv*(yv-mu))[:,None]*Xm; A=Xm.T@(Xm*(wv*mu*(1-mu))[:,None]); B,npsu,nstr,lonely=taylor_cov(score,strata,psu)
    Ai=np.linalg.inv(A); V=Ai@B@Ai; se=np.sqrt(np.diag(V)); df_=npsu-nstr; tq=stats.t.ppf(.975,df_); b_=fit.params.values
    out=pd.DataFrame({'AOR':np.exp(b_),'LCL':np.exp(b_-tq*se),'UCL':np.exp(b_+tq*se),'p':2*stats.t.sf(np.abs(b_/se),df_)},index=Zc.columns)
    return out, df_, lonely

In [ ]:
# A3. Table 2 and regional prevalence with design-based 95% CIs
def tab(var, labels):
    rows=[]
    for v,l in labels.items():
        m = (X[var]==v).values
        if m.sum():
            p,se,lo,hi = svy_mean(y[m],w[m],X.stratum[m],X.psu[m]); rows.append((l,int(m.sum()),round(100*p,1),f'{100*lo:.1f}–{100*hi:.1f}'))
    return rows
p0,se0,lo0,hi0 = svy_mean(y,w,X.stratum,X.psu); res['prev_ci']=[100*lo0,100*hi0]
T=[('All births',len(X),round(100*p0,1),f'{100*lo0:.1f}–{100*hi0:.1f}')]
T+=tab('rural',{0:'Urban',1:'Rural'})+tab('educ',{0:'No education',1:'Primary',2:'Secondary',3:'Higher'})+tab('wealth',{1:'Poorest',2:'Poorer',3:'Middle',4:'Richer',5:'Richest'})
T+=tab('anc4',{0:'<4 ANC visits',1:'>=4 ANC visits'})+tab('wanted',{1:'Pregnancy wanted then',2:'wanted later',3:'not wanted'})+tab('insured',{0:'No insurance',1:'Insured'})
show_table(pd.DataFrame(T,columns=['Characteristic','n (unweighted)','Home delivery % (weighted)','95% CI (design-based)']),'table2_descriptives.csv','Table 2. Home delivery by characteristic',index=False)
rn={1:'Tigray',2:'Afar',3:'Amhara',4:'Oromia',5:'Somali',6:'Benishangul-Gumuz',7:'Central Ethiopia',8:'Sidama',9:'South West Ethiopia',10:'South Ethiopia',12:'Gambella',13:'Harari',14:'Addis Ababa',15:'Dire Dawa'}
rows=[]
for r_,nm_ in rn.items():
    m=(X.region==r_).values; p,se,lo,hi=svy_mean(y[m],w[m],X.stratum[m],X.psu[m]); rows.append((nm_,int(m.sum()),100*p,100*lo,100*hi))
rp=pd.DataFrame(rows,columns=['reg','n','home_w','lcl','ucl']).sort_values('home_w',ascending=False)
show_table(rp.round(1),'region_prevalence.csv','Regional home-delivery prevalence, weighted % with design-based 95% CI',index=False)

## B. Spatial cross-validation design
Cluster coordinates are projected to **UTM zone 37N (EPSG:32637, metres)**. Ten spatial blocks are formed with k-means on the projected coordinates (so that blocks reflect true distances); blocks are then assigned to five outer folds (`GroupKFold`), so every cluster in a block is held out together.

In [ ]:
from sklearn.model_selection import StratifiedKFold, GroupKFold
from sklearn.cluster import KMeans
from scipy.spatial import cKDTree
cl_xy = gpd.GeoDataFrame(X.groupby('cluster')[['lat','lon']].first(), geometry=gpd.points_from_xy(X.groupby('cluster').lon.first(),X.groupby('cluster').lat.first()),crs=4326).to_crs(32637)
cl_xy['x']=cl_xy.geometry.x/1000; cl_xy['y']=cl_xy.geometry.y/1000          # kilometres
def make_blocks(k, seed=1): return pd.Series(KMeans(k,random_state=seed,n_init=10).fit(cl_xy[['x','y']].values).labels_,index=cl_xy.index)
def grid_blocks(size): return pd.Series(((cl_xy.x//size).astype(int).astype(str)+'_'+(cl_xy.y//size).astype(int).astype(str)),index=cl_xy.index).astype('category').cat.codes
blk = make_blocks(10); X['block']=X.cluster.map(blk).values; groups=X.block.values
def folds_from_groups(grp, seed=None, n=5):
    "Return list of (train_idx, test_idx). seed=None -> sklearn GroupKFold; otherwise blocks are randomly assigned to folds."
    if seed is None: return list(GroupKFold(n).split(X,y,grp))
    ug=np.unique(grp); rng=np.random.RandomState(seed); perm=rng.permutation(ug); fold_of={g_:i%n for i,g_ in enumerate(perm)}; f=np.array([fold_of[g_] for g_ in grp])
    return [(np.where(f!=i)[0],np.where(f==i)[0]) for i in range(n)]
sp_folds = folds_from_groups(groups)
rnd_folds = list(StratifiedKFold(5,shuffle=True,random_state=7).split(X,y))
fold_of_row = np.zeros(len(X),int)
for i,(tr,te) in enumerate(sp_folds): fold_of_row[te]=i
X['fold']=fold_of_row
def min_dist_km(folds):
    "Distance (km) from each test cluster to the nearest training cluster"
    out=[]; cid=X.cluster.values
    for tr,te in folds:
        trc=np.unique(cid[tr]); tec=np.unique(cid[te]); tree=cKDTree(cl_xy.loc[trc,['x','y']].values); out+=tree.query(cl_xy.loc[tec,['x','y']].values)[0].tolist()
    return np.array(out)
dd=pd.DataFrame({'Random 5-fold':pd.Series(min_dist_km(rnd_folds)).describe(),'Spatial block 5-fold':pd.Series(min_dist_km(sp_folds)).describe()}).round(1)
show_table(dd,'table_s2_train_test_distance.csv','Distance (km) from each test cluster to the nearest training cluster')
print('Clusters per block:', blk.value_counts().sort_index().tolist(), '| births per fold:', np.bincount(fold_of_row).tolist())

## C. Models: fixed hyper-parameters, random vs spatial validation

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, brier_score_loss, recall_score, precision_score, average_precision_score
import xgboost as xgb, lightgbm as lgb, shap
Xf=X[feat].copy(); Xf['Travel_Times']=Xf.Travel_Times.replace(-9999,np.nan)
Xf['log_pop']=np.log1p(Xf.All_Population_Count_2020); Xf['log_travel']=np.log1p(Xf.Travel_Times.clip(lower=0)); Xf['log_light']=np.log1p(Xf.Nightlights_Composite.clip(lower=0))
Xf=Xf.drop(columns=['All_Population_Count_2020','Travel_Times','Nightlights_Composite'])
Xlin=pd.get_dummies(Xf,columns=['region','religion'],drop_first=True).astype(float); Xtree=Xf.astype(float)
print('Predictors: tree models', Xtree.shape[1], '| logistic regression (dummy-coded)', Xlin.shape[1])
PARAMS = {'Logistic regression':'L2-penalised, C=0.5, median imputation + standardisation inside each fold',
 'Random forest':'500 trees, min_samples_leaf=5, max_features=sqrt, median imputation inside each fold',
 'XGBoost':'400 trees, max_depth=4, learning_rate=0.03, subsample=0.8, colsample_bytree=0.7, min_child_weight=5, reg_lambda=2; native missing handling',
 'LightGBM':'400 trees, num_leaves=15, learning_rate=0.03, subsample=0.8 (every iteration), colsample_bytree=0.7, min_child_samples=20, reg_lambda=2; native missing handling'}
show_table(pd.Series(PARAMS,name='Configuration').to_frame(),'table_s3_parameters.csv','Model configurations (fixed)')
models={
 'Logistic regression':lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(C=0.5,max_iter=2000)),
 'Random forest':lambda: make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(500,min_samples_leaf=5,n_jobs=-1,random_state=1)),
 'XGBoost':lambda: xgb.XGBClassifier(n_estimators=400,max_depth=4,learning_rate=0.03,subsample=0.8,colsample_bytree=0.7,min_child_weight=5,reg_lambda=2,eval_metric='logloss',n_jobs=-1,random_state=1),
 'LightGBM':lambda: lgb.LGBMClassifier(n_estimators=400,num_leaves=15,learning_rate=0.03,subsample=0.8,subsample_freq=1,colsample_bytree=0.7,min_child_samples=20,reg_lambda=2,verbose=-1,n_jobs=-1,random_state=1)}
design = lambda name: Xlin if name=='Logistic regression' else Xtree
def metrics(yt,p,sw=None):
    pr=(p>=0.5).astype(int)
    return dict(AUC=roc_auc_score(yt,p,sample_weight=sw),Accuracy=accuracy_score(yt,pr,sample_weight=sw),Sensitivity=recall_score(yt,pr,sample_weight=sw),Precision=precision_score(yt,pr,sample_weight=sw),F1=f1_score(yt,pr,sample_weight=sw),Brier=brier_score_loss(yt,p,sample_weight=sw),PR_AUC=average_precision_score(yt,p,sample_weight=sw))
oof={}; fold_auc=[]; rows=[]
for scheme,folds in [('Random 5-fold',rnd_folds),('Spatial block 5-fold',sp_folds)]:
    for name,mk in models.items():
        p=np.zeros(len(y)); XX=design(name)
        for i,(tr,te) in enumerate(folds):
            m=mk(); m.fit(XX.iloc[tr],y[tr]); p[te]=m.predict_proba(XX.iloc[te])[:,1]; fold_auc.append(dict(Validation=scheme,Model=name,Fold=i+1,AUC=roc_auc_score(y[te],p[te]),n_test=len(te)))
        oof[(scheme,name)]=p; r=metrics(y,p); r.update(Validation=scheme,Model=name); rows.append(r); print(f"{scheme:22s} {name:20s} AUC={r['AUC']:.3f} Brier={r['Brier']:.3f}")
perf=pd.DataFrame(rows)[['Validation','Model','AUC','Accuracy','Sensitivity','Precision','F1','Brier','PR_AUC']].round(3)
show_table(perf,'table4_model_performance_unweighted.csv','Table 4a. Out-of-fold performance (unweighted, threshold 0.5)',index=False)

In [ ]:
# C2. Survey-weighted performance (population-representative sensitivity analysis)
rows=[]
for (scheme,name),p in oof.items():
    r=metrics(y,p,w); r.update(Validation=scheme,Model=name); rows.append(r)
perf_w=pd.DataFrame(rows)[['Validation','Model','AUC','Accuracy','Sensitivity','Precision','F1','Brier','PR_AUC']].round(3)
show_table(perf_w,'table4b_model_performance_weighted.csv','Table 4b. Out-of-fold performance weighted by DHS sample weights',index=False)
# C3. Fold-specific performance and variability
fa=pd.DataFrame(fold_auc); fsum=fa.groupby(['Validation','Model']).AUC.agg(['mean','std','min','max']).round(3)
show_table(fa.pivot_table(index=['Validation','Model'],columns='Fold',values='AUC').round(3).join(fsum),'table_s4_fold_auc.csv','Fold-specific AUC (mean, SD, range across the five folds)')

In [ ]:
# C4. Cluster bootstrap 95% CIs for AUC and paired differences vs logistic regression
rng=np.random.RandomState(11); cid=X.cluster.values; uc=np.unique(cid); idx_by_c={c:np.where(cid==c)[0] for c in uc}; NB=500
boot_idx=[np.concatenate([idx_by_c[c] for c in rng.choice(uc,len(uc),replace=True)]) for _ in range(NB)]
rows=[]; bs={}
for key,p in oof.items(): bs[key]=np.array([roc_auc_score(y[ix],p[ix]) for ix in boot_idx])
for (scheme,name),a in bs.items():
    lo,hi=np.percentile(a,[2.5,97.5]); row=dict(Validation=scheme,Model=name,AUC=round(roc_auc_score(y,oof[(scheme,name)]),3),CI=f'{lo:.3f}–{hi:.3f}')
    if name!='Logistic regression':
        dlt=a-bs[(scheme,'Logistic regression')]; l2,h2=np.percentile(dlt,[2.5,97.5]); row['ΔAUC vs LR (95% CI)']=f'{dlt.mean():+.3f} ({l2:+.3f} to {h2:+.3f})'; row['p (boot)']=round(2*min((dlt<=0).mean(),(dlt>=0).mean()),3)
    rows.append(row)
auc_ci=pd.DataFrame(rows); show_table(auc_ci,'table4c_auc_ci.csv',f'AUC with cluster-bootstrap 95% CI ({NB} resamples) and paired difference vs logistic regression',index=False)

In [ ]:
# C5. Figure: AUC with 95% CI
fig,ax=plt.subplots(figsize=(7,4.6)); mods=list(models)
for j,(sc,col) in enumerate([('Random 5-fold','#2c7fb8'),('Spatial block 5-fold','#d95f0e')]):
    for i,mn in enumerate(mods):
        a=bs[(sc,mn)]; lo,hi=np.percentile(a,[2.5,97.5]); v=roc_auc_score(y,oof[(sc,mn)]); xpos=i+(j-.5)*.28
        ax.errorbar(xpos,v,yerr=[[v-lo],[hi-v]],fmt='o',color=col,capsize=3,label=sc if i==0 else None); ax.text(xpos,hi+.004,f'{v:.3f}',ha='center',fontsize=7)
ax.set_xticks(range(4)); ax.set_xticklabels([m.replace(' ','\n') for m in mods]); ax.set_ylabel('AUC (cluster-bootstrap 95% CI)'); ax.set_ylim(.78,.92); ax.legend(); ax.grid(alpha=.3)
ax.set_title('Discrimination under random vs spatial block cross-validation',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig07_auc_ci.png')

### C6. Calibration and precision–recall (spatial block validation)

In [ ]:
def logit(p): p=np.clip(p,1e-6,1-1e-6); return np.log(p/(1-p))
rows=[]
for name in models:
    p=oof[('Spatial block 5-fold',name)]; L=logit(p)
    slope=sm.Logit(y,sm.add_constant(L)).fit(disp=0).params[1]
    citl=sm.GLM(y,np.ones((len(y),1)),family=sm.families.Binomial(),offset=L).fit().params[0]
    rows.append(dict(Model=name,Calibration_intercept=round(citl,3),Calibration_slope=round(slope,3),Brier=round(brier_score_loss(y,p),3),PR_AUC=round(average_precision_score(y,p),3)))
cal=pd.DataFrame(rows); show_table(cal,'table4d_calibration.csv','Calibration intercept (calibration-in-the-large), slope, Brier and PR-AUC under spatial block validation',index=False)
fig,ax=plt.subplots(figsize=(6.2,6)); cols=['#444','#2c7fb8','#d95f0e','#31a354']
for (name,c) in zip(models,cols):
    p=oof[('Spatial block 5-fold',name)]; q=pd.qcut(p,10,duplicates='drop'); gdf_=pd.DataFrame({'p':p,'y':y}).groupby(q).mean(); ax.plot(gdf_.p,gdf_.y,'o-',color=c,label=name,ms=4)
ax.plot([0,1],[0,1],'k--',lw=1); ax.set_xlabel('Mean predicted probability (decile bins)'); ax.set_ylabel('Observed proportion home delivery'); ax.legend(fontsize=8); ax.grid(alpha=.3)
ax.set_title('Calibration under spatial block cross-validation',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig08_calibration.png')
from sklearn.metrics import precision_recall_curve
fig,ax=plt.subplots(figsize=(6.2,6))
for (name,c) in zip(models,cols):
    p=oof[('Spatial block 5-fold',name)]; pr_,rc_,_=precision_recall_curve(y,p); ax.plot(rc_,pr_,color=c,label=f'{name} (PR-AUC {average_precision_score(y,p):.3f})')
ax.axhline(y.mean(),color='k',ls='--',lw=1,label=f'Prevalence {y.mean():.3f}'); ax.set_xlabel('Recall (sensitivity)'); ax.set_ylabel('Precision'); ax.legend(fontsize=8,loc='upper right'); ax.grid(alpha=.3)
ax.set_title('Precision–recall curves under spatial block validation',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig09_pr_curves.png')

## D. Sensitivity of spatial validation (blocks, buffers, fold assignment)

In [ ]:
def cv_auc(name, folds, buffer_km=0, Xmat=None, ysub=None):
    XX=design(name) if Xmat is None else Xmat; p=np.full(len(y),np.nan); cidv=X.cluster.values
    for tr,te in folds:
        if buffer_km>0:
            tec=np.unique(cidv[te]); tree=cKDTree(cl_xy.loc[tec,['x','y']].values); dcl=pd.Series(tree.query(cl_xy[['x','y']].values)[0],index=cl_xy.index); tr=tr[(dcl.loc[cidv[tr]].values>=buffer_km)]
        m=models[name](); m.fit(XX.iloc[tr],y[tr]); p[te]=m.predict_proba(XX.iloc[te])[:,1]
    ok=~np.isnan(p); return roc_auc_score(y[ok],p[ok])
rows=[]
for label,grp,seed in [('k-means k=5',make_blocks(5),None),('k-means k=10 (primary)',blk,None),('k-means k=15',make_blocks(15),None),('k-means k=20',make_blocks(20),None),('Square grid 100 km',grid_blocks(100),None),('Square grid 200 km',grid_blocks(200),None)]:
    gr=X.cluster.map(grp).values; fl=folds_from_groups(gr,seed)
    rows.append(dict(Configuration=label,**{n:round(cv_auc(n,fl),3) for n in ['Logistic regression','XGBoost']})); print(rows[-1])
for s in range(1,6):
    fl=folds_from_groups(groups,seed=s); rows.append(dict(Configuration=f'k=10 blocks, random fold assignment seed {s}',**{n:round(cv_auc(n,fl),3) for n in ['Logistic regression','XGBoost']})); print(rows[-1])
for bkm in [25,50,100]:
    rows.append(dict(Configuration=f'k=10 blocks + {bkm} km training buffer',**{n:round(cv_auc(n,sp_folds,bkm),3) for n in ['Logistic regression','XGBoost']})); print(rows[-1])
sens_cv=pd.DataFrame(rows); show_table(sens_cv,'table_s5_spatial_cv_sensitivity.csv','Sensitivity of spatial-validation AUC to block design, fold assignment and buffering',index=False)
seedrows=sens_cv[sens_cv.Configuration.str.contains('seed')]; print('Mean ± SD across 5 fold assignments:', {n:(round(seedrows[n].mean(),3),round(seedrows[n].std(),3)) for n in ['Logistic regression','XGBoost']})

In [ ]:
# D2. Figure: spatial blocks and outer folds
ne_path=os.path.join(DATA_DIR,'eth_ne.gpkg')
import urllib.request
if not os.path.exists(ne_path):
    print('Downloading Natural Earth admin-1 (one-off, ~40 MB)...'); urllib.request.urlretrieve('https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/geojson/ne_10m_admin_1_states_provinces.geojson','/tmp/ne.geojson')
    gpd.read_file('/tmp/ne.geojson').query("admin=='Ethiopia'").to_file(ne_path)
ne=gpd.read_file(ne_path)
from shapely.geometry import MultiPoint, box
from shapely.ops import voronoi_diagram
nm={'Somali':'Somali','Afar':'Afar','Amhara':'Amhara','Tigray':'Tigray','Benshangul-Gumaz':'Benishangul-Gumuz','Gambela Peoples':'Gambella','Oromiya':'Oromia','Dire Dawa':'Dire Dawa','Addis Ababa':'Addis Ababa','Harari People':'Harari'}
ne['reg']=ne.name.map(nm); gps=gpd.read_file(os.path.join(DATA_DIR,'ETGE8AFL.shp')); south=['Sidama','Central Ethiopia','South Ethiopia','South West Ethiopia']
off=os.path.join(DATA_DIR,'eth_admin1_official.gpkg')
if os.path.exists(off):
    reg=gpd.read_file(off)[['reg','geometry']].to_crs(4326); BOUND='official 2023 boundaries (user-supplied)'
else:
    snnp=ne[ne.name.str.startswith('Southern')].geometry.iloc[0].buffer(0); pts=gps[gps.DHSREGNA.isin(south)]
    vg=gpd.GeoDataFrame(geometry=list(voronoi_diagram(MultiPoint(list(pts.geometry)),envelope=box(30,2,45,16)).geoms),crs=4326)
    vg=gpd.sjoin(vg,pts[['DHSREGNA','geometry']],predicate='contains').dissolve(by='DHSREGNA').reset_index(); vg['geometry']=vg.geometry.intersection(snnp); vg=vg.rename(columns={'DHSREGNA':'reg'})[['reg','geometry']]
    reg=gpd.GeoDataFrame(pd.concat([ne[ne.reg.notna()][['reg','geometry']],vg],ignore_index=True),crs=4326); BOUND='Natural Earth (10 regions) + approximate delineation of 4 ex-SNNPR regions'
reg['geometry']=reg.geometry.buffer(0); print('Boundary source:',BOUND)
pal=['#cfe3f2','#f6d7b0','#d5e8c4','#f2c9c9','#e3d4ec','#fbf0b5','#c9e8e2','#e8d3c1','#d9dcee','#efd0e1','#d3e5d9','#f1e1c4','#cfd8e8','#e0e0e0']; reg['c']=[pal[i%len(pal)] for i in range(len(reg))]
def base(ax,fill=True):
    (reg.plot(ax=ax,color=reg.c,edgecolor='#555',linewidth=0.8) if fill else reg.plot(ax=ax,color='#f3f3f3',edgecolor='#777',linewidth=0.7))
    ax.set_xlim(32.8,48.2); ax.set_ylim(3.2,15.2); ax.set_xticks(range(33,48,3)); ax.set_yticks(range(4,16,2)); ax.set_xticklabels([f'{x}°E' for x in range(33,48,3)],fontsize=7); ax.set_yticklabels([f'{y_}°N' for y_ in range(4,16,2)],fontsize=7); ax.grid(alpha=.25,lw=.4)
def northarrow(ax): ax.annotate('N',xy=(47.4,14.7),xytext=(47.4,13.5),ha='center',fontsize=11,fontweight='bold',arrowprops=dict(arrowstyle='-|>',color='k',lw=1.5))
def scalebar(ax): ax.plot([33.3,35.1],[3.6,3.6],'k',lw=3); ax.text(34.2,3.72,'≈200 km',ha='center',fontsize=7)
fig,ax=plt.subplots(figsize=(7.8,7)); base(ax,False); fcol=['#1b9e77','#d95f02','#7570b3','#e7298a','#66a61e']
cb=X.groupby('cluster').agg(lat=('lat','first'),lon=('lon','first'),block=('block','first'),fold=('fold','first'))
for f_ in range(5):
    s=cb[cb.fold==f_]; ax.scatter(s.lon,s.lat,c=fcol[f_],s=14,edgecolor='k',linewidth=.2,label=f'Outer fold {f_+1} ({len(s)} clusters)',zorder=3)
for bk,s in cb.groupby('block'): ax.text(s.lon.mean(),s.lat.mean(),f'B{bk+1}',fontsize=8,fontweight='bold',ha='center',va='center',path_effects=halo,zorder=5)
ax.legend(fontsize=7,loc='lower right'); northarrow(ax); ax.set_title('Spatial validation blocks (k-means, UTM 37N) and outer folds',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig06_spatial_blocks.png')

## E. Missing-data strategy sensitivity and nested hyper-parameter tuning

In [ ]:
rows=[]; cc_idx=np.where(cc.values)[0]
def cv_generic(mk, XX, rows_idx=None):
    p=np.full(len(y),np.nan)
    for tr,te in sp_folds:
        if rows_idx is not None: tr=np.intersect1d(tr,rows_idx); te=np.intersect1d(te,rows_idx)
        m=mk(); m.fit(XX.iloc[tr],y[tr]); p[te]=m.predict_proba(XX.iloc[te])[:,1]
    ok=~np.isnan(p); return roc_auc_score(y[ok],p[ok]), int(ok.sum())
imp_xgb=lambda: make_pipeline(SimpleImputer(strategy='median'),models['XGBoost']())
for lab,mk,XX,ri in [('XGBoost: native missing handling (primary)',models['XGBoost'],Xtree,None),('XGBoost: median imputation inside folds',imp_xgb,Xtree,None),('XGBoost: complete cases only',models['XGBoost'],Xtree,cc_idx),
                     ('Logistic regression: median imputation inside folds (primary)',models['Logistic regression'],Xlin,None),('Logistic regression: complete cases only',models['Logistic regression'],Xlin,cc_idx)]:
    a,n=cv_generic(mk,XX,ri); rows.append(dict(Strategy=lab,AUC_spatial_CV=round(a,3),n_births=n)); print(rows[-1])
show_table(pd.DataFrame(rows),'table_s6_missing_data_sensitivity.csv','Missing-data strategy sensitivity (spatial block CV)',index=False)
# Sensitivity: leave-one-out, all-women cluster context variables
Xs=Xtree.copy(); Xs['c_poor']=X.c_poor_loo.values; Xs['c_noeduc']=X.c_noeduc_loo.values
a_loo,_=cv_generic(models['XGBoost'],Xs); print(f'XGBoost AUC with leave-one-out all-women cluster variables: {a_loo:.3f}  (primary {roc_auc_score(y,oof[("Spatial block 5-fold","XGBoost")]):.3f})')
res['auc_loo_context']=float(a_loo)

In [ ]:
# E2. Nested hyper-parameter tuning: inner 3-fold grouped CV on blocks inside each outer training set (spatial outer folds)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform, loguniform
spaces={
 'Logistic regression':(lambda: make_pipeline(SimpleImputer(strategy='median'),StandardScaler(),LogisticRegression(max_iter=3000)),{'logisticregression__C':loguniform(1e-3,1e2)},8),
 'Random forest':(lambda: make_pipeline(SimpleImputer(strategy='median'),RandomForestClassifier(200,n_jobs=-1,random_state=1)),{'randomforestclassifier__min_samples_leaf':randint(2,15),'randomforestclassifier__max_features':uniform(0.1,0.5)},3),
 'XGBoost':(lambda: xgb.XGBClassifier(eval_metric='logloss',n_jobs=-1,random_state=1),{'n_estimators':randint(150,500),'max_depth':randint(2,7),'learning_rate':loguniform(0.01,0.12),'subsample':uniform(0.6,0.4),'colsample_bytree':uniform(0.5,0.5),'min_child_weight':randint(1,15),'reg_lambda':loguniform(0.5,20)},8),
 'LightGBM':(lambda: lgb.LGBMClassifier(verbose=-1,n_jobs=-1,random_state=1,subsample_freq=1),{'n_estimators':randint(150,500),'num_leaves':randint(4,40),'learning_rate':loguniform(0.01,0.12),'subsample':uniform(0.6,0.4),'colsample_bytree':uniform(0.5,0.5),'min_child_samples':randint(10,60),'reg_lambda':loguniform(0.5,20)},8)}
tuned={}; chosen=[]; t0=time.time()
for name,(mk,dist,nit) in spaces.items():
    XX=design(name); p=np.zeros(len(y))
    for i,(tr,te) in enumerate(sp_folds):
        inner=list(GroupKFold(3).split(XX.iloc[tr],y[tr],groups[tr]))
        s=RandomizedSearchCV(mk(),dist,n_iter=nit,cv=inner,scoring='roc_auc',random_state=i,refit=True,n_jobs=1); s.fit(XX.iloc[tr],y[tr]); p[te]=s.predict_proba(XX.iloc[te])[:,1]
        chosen.append(dict(Model=name,Outer_fold=i+1,Inner_AUC=round(s.best_score_,3),**{k.split('__')[-1]:(round(v,4) if isinstance(v,float) else v) for k,v in s.best_params_.items()}))
    tuned[name]=p; print(f'{name:20s} nested-tuned spatial AUC = {roc_auc_score(y,p):.3f}  ({time.time()-t0:.0f}s elapsed)')
nt=pd.DataFrame([dict(Model=n,Fixed_AUC=round(roc_auc_score(y,oof[('Spatial block 5-fold',n)]),3),Nested_tuned_AUC=round(roc_auc_score(y,tuned[n]),3),Fixed_Brier=round(brier_score_loss(y,oof[('Spatial block 5-fold',n)]),3),Nested_tuned_Brier=round(brier_score_loss(y,tuned[n]),3)) for n in models])
show_table(nt,'table_s7_nested_tuning.csv','Fixed vs nested-tuned models (spatial block outer CV, inner grouped 3-fold CV)',index=False); show_table(pd.DataFrame(chosen),'table_s8_selected_hyperparameters.csv','Selected hyper-parameters per outer fold',index=False)

## F. Interpretation: SHAP (full-sample explanation and held-out explanation)
The beeswarm and dependence plots explain the **XGBoost model fitted on the full sample** (an explanation of the fitted model, not a prediction test). As a check, SHAP values were also computed for each **held-out spatial fold** with the model trained on the other folds.

In [ ]:
best=models['XGBoost'](); best.fit(Xtree,y); sv=shap.TreeExplainer(best).shap_values(Xtree)
imp=pd.Series(np.abs(sv).mean(0),index=Xtree.columns).sort_values(ascending=False)
sv_cv=np.zeros_like(sv)
for tr,te in sp_folds:
    m=models['XGBoost'](); m.fit(Xtree.iloc[tr],y[tr]); sv_cv[te]=shap.TreeExplainer(m).shap_values(Xtree.iloc[te])
imp_cv=pd.Series(np.abs(sv_cv).mean(0),index=Xtree.columns)
rho=stats.spearmanr(imp.values,imp_cv.reindex(imp.index).values)[0]; print(f'Spearman rank correlation between full-sample and held-out SHAP importance: {rho:.3f}'); res['shap_rank_rho']=float(rho)
show_table(pd.DataFrame({'mean_abs_SHAP_full':imp.round(3),'mean_abs_SHAP_heldout_folds':imp_cv.reindex(imp.index).round(3)}).head(20),'shap_importance.csv','Top-20 features by mean |SHAP| (log-odds)')
names={'anc_visits':'ANC visits (n)','c_poor':'Cluster share poor','wealth':'Wealth quintile','dist_min':'Minutes to facility (self-reported)','parity':'Parity','tetanus':'Tetanus injections','region':'Region','educ':'Education','rural':'Rural residence','log_travel':'Log travel time to city','mobile':'Owns mobile phone','Rainfall_2020':'Annual rainfall','bank':'Bank account','Land_Surface_Temperature_2020':'Land surface temp.','c_noeduc':'Cluster share no education'}
idx=np.argsort(np.abs(sv).mean(0))[::-1][:15]
plt.figure(figsize=(8,6.2)); shap.summary_plot(sv[:,idx],Xtree.iloc[:,idx].rename(columns=names),show=False,max_display=15,plot_size=None); plt.title('SHAP summary (XGBoost): drivers of home delivery',fontsize=10,fontweight='bold',loc='left'); show_fig(plt.gcf(),'fig10_shap_beeswarm.png')
for k,(fv,fn) in enumerate([('anc_visits','fig11_shap_dep_anc'),('c_poor','fig12_shap_dep_cluster_poverty'),('wealth','fig13_shap_dep_wealth'),('dist_min','fig14_shap_dep_distance'),('parity','fig15_shap_dep_parity')]):
    plt.figure(figsize=(6,4.4)); shap.dependence_plot(fv,sv,Xtree,show=False,interaction_index=None); plt.gca().set_xlabel(names.get(fv,fv)); plt.gca().set_ylabel('SHAP value (log-odds of home delivery)')
    plt.title(f'SHAP dependence: {names.get(fv,fv)}',fontsize=10,fontweight='bold',loc='left'); show_fig(plt.gcf(),fn+'.png')

## G. Design-based logistic regression (Taylor-series linearisation)
Weights (`v005`), strata (`v022`, 27 strata) and primary sampling units (`v021`, 794 PSUs) are all used; 95% CIs and p-values use a t distribution with (PSUs − strata) degrees of freedom. The previous cluster-robust-only intervals are printed for comparison.

In [ ]:
sel=['age','educ','wealth','rural','married','parity','anc4','wanted','news','insured','dist_min','female_head','c_noeduc','c_poor']
D=X[sel+['region']].copy(); D['dist10']=D.dist_min/10; D['age5']=D.age/5; D=D.drop(columns=['dist_min','age'])
Z=pd.concat([D.drop(columns=['educ','wealth','region']),pd.get_dummies(D.educ.astype(int),prefix='educ',drop_first=True),pd.get_dummies(D.wealth.astype(int),prefix='wealth',drop_first=True),pd.get_dummies(D.region.astype(int),prefix='reg',drop_first=True)],axis=1).astype(float)
ok=Z.notna().all(axis=1).values; print('Births with complete data for regression:',int(ok.sum()))
aor,dfr,lonely=svy_logit(Z[ok],y[ok],w[ok],X.stratum[ok],X.psu[ok]); print('Design df =',dfr,'| strata with a single PSU skipped:',lonely)
wn=w[ok]/w[ok].mean(); glm=sm.GLM(y[ok],sm.add_constant(Z[ok]),family=sm.families.Binomial(),var_weights=wn).fit(cov_type='cluster',cov_kwds={'groups':X.cluster[ok].values})
cmp_=pd.DataFrame({'AOR':aor.AOR.round(3),'design LCL':aor.LCL.round(3),'design UCL':aor.UCL.round(3),'design p':aor.p.round(4),'cluster-robust LCL':np.exp(glm.conf_int()[0]).round(3),'cluster-robust UCL':np.exp(glm.conf_int()[1]).round(3)})
show_table(cmp_,'table5_aor_design_based.csv','Table 5. Adjusted odds ratios (design-based Taylor linearisation; reference: no education, poorest, Tigray)')
res['reg_n']=int(ok.sum()); res['reg_df']=int(dfr)

## H. Spatial statistics and sensitivity

In [ ]:
from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local
from esda.getisord import G_Local
from statsmodels.stats.multitest import multipletests
cs=X.groupby('cluster').apply(lambda s:pd.Series({'prev':np.average(s.home,weights=s.w),'raw_prev':s.home.mean(),'events':s.home.sum(),'n':len(s),'lat':s.lat.iloc[0],'lon':s.lon.iloc[0],'rural':s.rural.iloc[0],'region':s.region.iloc[0]})).reset_index()
gdf=gpd.GeoDataFrame(cs,geometry=gpd.points_from_xy(cs.lon,cs.lat),crs=4326).to_crs(32637)
def local_stats(gd,k,seed=1,val='prev'):
    W=KNN.from_dataframe(gd,k=k); W.transform='r'; mi=Moran(gd[val].values,W,permutations=999)
    lm=Moran_Local(gd[val].values,W,permutations=999,seed=seed); q=lm.q; p_l=lm.p_sim
    Wb=KNN.from_dataframe(gd,k=k); gi=G_Local(gd[val].values,Wb,transform='B',star=True,permutations=999,seed=seed)
    return W,mi,lm,gi
rows=[]
for k in [4,6,8,10]:
    W_,mi_,lm_,gi_=local_stats(gdf,k); q=lm_.q; sig=lm_.p_sim<0.05
    fdr_l=multipletests(lm_.p_sim,alpha=0.05,method='fdr_bh')[0]; fdr_g=multipletests(gi_.p_sim,alpha=0.05,method='fdr_bh')[0]
    rows.append(dict(k=k,Moran_I=round(mi_.I,3),z=round(mi_.z_sim,1),p=mi_.p_sim,LISA_HH=int((sig&(q==1)).sum()),LISA_LL=int((sig&(q==3)).sum()),LISA_HH_FDR=int((fdr_l&(q==1)).sum()),LISA_LL_FDR=int((fdr_l&(q==3)).sum()),
                     Gi_hot=int(((gi_.Zs>0)&(gi_.p_sim<0.05)).sum()),Gi_cold=int(((gi_.Zs<0)&(gi_.p_sim<0.05)).sum()),Gi_hot_FDR=int(((gi_.Zs>0)&fdr_g).sum()),Gi_cold_FDR=int(((gi_.Zs<0)&fdr_g).sum())))
ksens=pd.DataFrame(rows); show_table(ksens,'table_s9_neighbourhood_sensitivity.csv','Neighbourhood sensitivity (k nearest neighbours) with and without Benjamini–Hochberg FDR (q = 0.05)',index=False)
# Cluster-size effects
from esda.smoothing import Empirical_Bayes
rows=[]
W8=KNN.from_dataframe(gdf,k=8); W8.transform='r'
rows.append(dict(Specification='Primary: weighted cluster prevalence, all 794 clusters',I=round(Moran(gdf.prev.values,W8,permutations=999).I,3)))
big=gdf[gdf.n>=8].copy(); Wb_=KNN.from_dataframe(big,k=8); Wb_.transform='r'; rows.append(dict(Specification=f'Clusters with ≥8 births only (n={len(big)})',I=round(Moran(big.prev.values,Wb_,permutations=999).I,3)))
eb=Empirical_Bayes(gdf.events.values.reshape(-1,1),gdf.n.values.reshape(-1,1)).r.ravel(); rows.append(dict(Specification='Empirical-Bayes smoothed unweighted prevalence',I=round(Moran(eb,W8,permutations=999).I,3)))
rows.append(dict(Specification='Unweighted cluster prevalence',I=round(Moran(gdf.raw_prev.values,W8,permutations=999).I,3)))
show_table(pd.DataFrame(rows),'table_s10_cluster_size_sensitivity.csv',"Global Moran's I under alternative handling of unequal cluster sample size",index=False)
# primary k = 8 classification
W_,mi,lm,gi=local_stats(gdf,8); q=lm.q; sig=lm.p_sim<0.05
print(f"PRIMARY k=8: Moran's I = {mi.I:.3f}, z = {mi.z_sim:.1f}, p = {mi.p_sim:.3f}"); res.update(moran_I=float(mi.I),moran_z=float(mi.z_sim),moran_p=float(mi.p_sim))
gdf['lisa']=np.where(~sig,'Not significant',np.select([q==1,q==3,q==2,q==4],['High-High','Low-Low','Low-High','High-Low'],'NS')); gdf['gi_z']=gi.Zs; gdf['gi_p']=gi.p_sim
gdf['gi_class']=np.select([(gdf.gi_z>=2.58)&(gdf.gi_p<0.05),(gdf.gi_z>=1.96)&(gdf.gi_p<0.05),(gdf.gi_z>=1.65)&(gdf.gi_p<0.10),(gdf.gi_z<=-2.58)&(gdf.gi_p<0.05),(gdf.gi_z<=-1.96)&(gdf.gi_p<0.05),(gdf.gi_z<=-1.65)&(gdf.gi_p<0.10)],['Hot 99%','Hot 95%','Hot 90%','Cold 99%','Cold 95%','Cold 90%'],'Not significant')
gdf['gi_fdr']=multipletests(gdf.gi_p,alpha=0.05,method='fdr_bh')[0]
print('LISA:',gdf.lisa.value_counts().to_dict()); print('Gi*:',gdf.gi_class.value_counts().to_dict())
cs2=gdf.to_crs(4326).drop(columns='geometry'); cs2['reg']=cs2.region.map(rn)
# residual autocorrelation (cluster-averaged out-of-fold residuals)
for key in [('Random 5-fold','XGBoost'),('Spatial block 5-fold','XGBoost'),('Spatial block 5-fold','Logistic regression')]:
    r_=pd.Series(y-oof[key]).groupby(X.cluster.values).mean().reindex(gdf.cluster.values).values; m2=Moran(r_,W8,permutations=999); print(f"Residual Moran's I {key}: {m2.I:.3f} (p={m2.p_sim:.3f})"); res['resid_'+key[0].split()[0]+'_'+key[1].split()[0]]=[float(m2.I),float(m2.p_sim)]
cs2['resid']=pd.Series(y-oof[('Spatial block 5-fold','XGBoost')]).groupby(X.cluster.values).mean().reindex(cs2.cluster.values).values
hc=pd.crosstab(cs2.reg,cs2.gi_class.str[:4]).rename(columns={'Cold':'Gi* cold','Hot':'Gi* hot','Not ':'not sig.'}); hc['Gi* hot (FDR)']=cs2[cs2.gi_fdr&(cs2.gi_z>0)].groupby('reg').size().reindex(hc.index).fillna(0).astype(int); hc['Gi* cold (FDR)']=cs2[cs2.gi_fdr&(cs2.gi_z<0)].groupby('reg').size().reindex(hc.index).fillna(0).astype(int)
hc['LISA HH']=cs2[cs2.lisa=='High-High'].groupby('reg').size().reindex(hc.index).fillna(0).astype(int); hc['LISA LL']=cs2[cs2.lisa=='Low-Low'].groupby('reg').size().reindex(hc.index).fillna(0).astype(int)
hc['mean OOF residual (obs − pred)']=cs2.groupby('reg').resid.mean().round(3)
show_table(hc,'table3_hotspots_by_region.csv','Hot/cold clusters by region (Gi* unadjusted and FDR-adjusted), LISA clusters and mean residual of the spatially validated XGBoost model')
cs2.to_csv(P('cluster_spatial.csv'),index=False)

## I. Maps (separate figures): study area, prevalence, hot spots, LISA, residuals

In [ ]:
caps={'Tigray':('Mekelle',13.497,39.475),'Afar':('Semera',11.79,41.00),'Amhara':('Bahir Dar',11.594,37.388),'Oromia':('Addis Ababa (Finfinne)',9.03,38.74),'Somali':('Jigjiga',9.35,42.80),'Benishangul-Gumuz':('Asosa',10.07,34.53),'Central Ethiopia':('Hosaina',7.55,37.85),'Sidama':('Hawassa',7.06,38.48),'South West Ethiopia':('Bonga',7.27,36.24),'South Ethiopia':('Wolaita Sodo',6.86,37.76),'Gambella':('Gambela',8.25,34.59),'Harari':('Harar',9.31,42.12),'Addis Ababa':('Addis Ababa',9.03,38.74),'Dire Dawa':('Dire Dawa',9.60,41.86)}
lab={'Tigray':(38.9,14.1),'Afar':(40.9,12.2),'Amhara':(37.7,12.3),'Oromia':(39.6,7.9),'Somali':(44.3,6.8),'Benishangul-Gumuz':(35.2,10.9),'Central Ethiopia':(37.0,8.15),'Sidama':(38.9,6.15),'South West Ethiopia':(36.0,7.0),'South Ethiopia':(37.2,5.7),'Gambella':(34.2,7.6)}
fig,ax=plt.subplots(figsize=(8.5,7.6)); base(ax)
for r,(x_,y_) in lab.items(): ax.text(x_,y_,r.upper().replace(' ','\n',1) if len(r)>12 else r.upper(),ha='center',va='center',fontsize=8,fontweight='bold',color='#222',path_effects=halo)
for t_,xy,xyt in [('DIRE DAWA',(41.85,9.58),(43.6,10.7)),('HARARI',(42.2,9.3),(44.0,9.2)),('ADDIS ABABA',(38.75,9.02),(36.6,9.7))]: ax.annotate(t_,xy=xy,xytext=xyt,fontsize=6.5,fontweight='bold',ha='center',arrowprops=dict(arrowstyle='-',lw=.6),path_effects=halo)
for r,(nmc,la,lo) in caps.items():
    if r=='Oromia': continue
    star=r=='Addis Ababa'; ax.plot(lo,la,marker='*' if star else 'o',ms=11 if star else 5.5,color='crimson' if star else 'k',mec='white',mew=.6,zorder=5)
    dx,dy=(0.18,-0.45) if star else {'Harari':(0.12,-0.38),'Dire Dawa':(-0.1,-0.4),'Hosaina':(-0.12,-0.32),'Sidama':(0.12,-0.28),'Wolaita Sodo':(0.12,-0.3)}.get(nmc,(0.12,0.1))
    ax.text(lo+dx,la+dy,'Addis Ababa\n(national capital;\nOromia seat)' if star else nmc,fontsize=6.8,ha='right' if r in('Dire Dawa','Hosaina') else 'left',style='italic',path_effects=halo,zorder=6)
ax.plot([],[],'o',color='k',ms=5,label='Regional capital'); ax.plot([],[],'*',color='crimson',ms=11,label='National capital'); ax.legend(loc='lower right',fontsize=8)
ax.set_title('Administrative regions of Ethiopia and regional capitals',fontsize=10,fontweight='bold',loc='left'); northarrow(ax); scalebar(ax); fig.text(0.01,0.005,'Boundaries: '+BOUND+'. WGS84.',fontsize=6,color='#444'); show_fig(fig,'fig01_study_area.png')
R=reg.merge(rp[['reg','home_w']],on='reg'); fig,ax=plt.subplots(figsize=(8,7)); base(ax,False)
R.plot(column='home_w',cmap='YlOrRd',vmin=0,vmax=70,edgecolor='#444',linewidth=.8,ax=ax,legend=True,legend_kwds=dict(label='Home delivery (%), survey-weighted',shrink=.6))
for _,r in R.iterrows():
    c=r.geometry.representative_point(); ax.text(c.x,c.y,f"{r.reg.replace(' ','\n',1) if len(r.reg)>12 else r.reg}\n{r.home_w:.1f}%",ha='center',va='center',fontsize=6,fontweight='bold',path_effects=halo)
northarrow(ax); scalebar(ax); ax.set_title('Regional prevalence of home delivery',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig02_regional_prevalence.png')
fig,ax=plt.subplots(figsize=(8,7)); base(ax,False); reg.boundary.plot(ax=ax,color='#888',lw=.5)
sc=ax.scatter(cs2.lon,cs2.lat,c=cs2.prev*100,cmap='YlOrRd',s=14+cs2.n,edgecolor='k',linewidth=.25,vmin=0,vmax=100,alpha=.9); plt.colorbar(sc,ax=ax,shrink=.6,label='Cluster home delivery (%)')
for r,(nmc,la,lo) in caps.items(): ax.plot(lo,la,'k^',ms=3.5)
northarrow(ax); scalebar(ax); ax.set_title('Cluster-level prevalence (displaced GPS locations)',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig03_cluster_prevalence.png')
cg={'Hot 99%':'#b2182b','Hot 95%':'#ef6548','Hot 90%':'#fdbb84','Not significant':'#d9d9d9','Cold 90%':'#c6dbef','Cold 95%':'#6baed6','Cold 99%':'#2166ac'}
lc={'High-High':'#d7191c','Low-Low':'#2b83ba','High-Low':'#fdae61','Low-High':'#abd9e9','Not significant':'#d9d9d9'}
for col,dct,ttl,lg,fn in [('gi_class',cg,'Getis-Ord Gi* hot and cold spots (k = 8)','Getis-Ord Gi*','fig04_gi_hotspots.png'),('lisa',lc,'Local Moran’s I clusters and outliers (k = 8)',"Local Moran's I",'fig05_lisa.png')]:
    fig,ax=plt.subplots(figsize=(8,7)); base(ax,False)
    for k_,c_ in dct.items():
        s=cs2[cs2[col]==k_]; ax.scatter(s.lon,s.lat,c=c_,s=18,edgecolor='k',linewidth=.2,label=f'{k_} (n={len(s)})',zorder=3)
    ax.legend(fontsize=7,loc='upper right',title=lg,title_fontsize=8)
    for r,(x_,y_) in lab.items(): ax.text(x_,y_,r,fontsize=5.5,ha='center',color='#555',style='italic',path_effects=halo)
    ax.set_title(ttl,fontsize=10,fontweight='bold',loc='left'); show_fig(fig,fn)
fig,ax=plt.subplots(figsize=(8,7)); base(ax,False); sc=ax.scatter(cs2.lon,cs2.lat,c=cs2.resid,cmap='RdBu_r',vmin=-.6,vmax=.6,s=14+cs2.n,edgecolor='k',linewidth=.25); plt.colorbar(sc,ax=ax,shrink=.6,label='Mean out-of-fold residual (observed − predicted)')
northarrow(ax); ax.set_title('Residuals of the spatially validated XGBoost model',fontsize=10,fontweight='bold',loc='left'); show_fig(fig,'fig16_residual_map.png')

## J. Summary and saved files

In [ ]:
json.dump(res,open(P('summary.json'),'w'),indent=1,default=str); print(json.dumps(res,indent=1,default=str)); print('\nFiles saved in',OUT); print(sorted(os.listdir(OUT)))